
# Jalrakshak-ML — Error Analysis

## Objectives
- Analyze Random Forest classification errors.
- Inspect class-wise performance and confusion matrix.
- Investigate misclassified High irrigation-need cases.
- Identify potential areas for model improvement.

In [1]:

from pathlib import Path

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "irrigation_prediction.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "irrigation_need_model.joblib"

TARGET = "Irrigation_Need"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nTarget distribution:")
print(df[TARGET].value_counts())

Dataset shape: (10000, 20)
Columns: ['Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']

Target distribution:
Irrigation_Need
Low       5864
Medium    3800
High       336
Name: count, dtype: int64


In [2]:

from sklearn.model_selection import train_test_split

TARGET = "Irrigation_Need"

NUMERIC_FEATURES = [
    "Soil_Moisture",
    "Temperature_C",
    "Humidity",
]

CATEGORICAL_FEATURES = [
    "Crop_Type",
    "Crop_Growth_Stage",
    "Soil_Type",
]

FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTest class distribution:")
print(y_test.value_counts())

print("\nTest class proportions:")
print(y_test.value_counts(normalize=True).round(4))


Training rows: 8000
Testing rows: 2000

Test class distribution:
Irrigation_Need
Low       1173
Medium     760
High        67
Name: count, dtype: int64

Test class proportions:
Irrigation_Need
Low       0.5865
Medium    0.3800
High      0.0335
Name: proportion, dtype: float64


In [3]:

model = joblib.load(MODEL_PATH)

y_pred = model.predict(X_test)

print("Actual test labels:", len(y_test))
print("Predictions generated:", len(y_pred))

print("\nPredicted class distribution:")
print(pd.Series(y_pred, name="Predicted_Irrigation_Need").value_counts())

print("\nFirst 10 predictions:")
print(
    pd.DataFrame({
        "Actual": y_test.reset_index(drop=True),
        "Predicted": y_pred,
    }).head(10)
)


Actual test labels: 2000
Predictions generated: 2000

Predicted class distribution:
Predicted_Irrigation_Need
Low       1168
Medium     716
High       116
Name: count, dtype: int64

First 10 predictions:
   Actual Predicted
0  Medium    Medium
1  Medium       Low
2     Low       Low
3  Medium    Medium
4  Medium    Medium
5     Low       Low
6  Medium       Low
7     Low       Low
8  Medium      High
9     Low       Low


In [4]:

from sklearn.metrics import confusion_matrix, classification_report

LABELS = ["Low", "Medium", "High"]

cm = confusion_matrix(y_test, y_pred, labels=LABELS)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in LABELS],
    columns=[f"Predicted_{label}" for label in LABELS],
)

print("Confusion Matrix:")
display(cm_df)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=LABELS,
        zero_division=0,
        digits=4,
    )
)

Confusion Matrix:


,Predicted_Low,Predicted_Medium,Predicted_High
Actual_Low,972,199,2
Actual_Medium,196,488,76
Actual_High,0,29,38



Classification Report:
              precision    recall  f1-score   support

         Low     0.8322    0.8286    0.8304      1173
      Medium     0.6816    0.6421    0.6612       760
        High     0.3276    0.5672    0.4153        67

    accuracy                         0.7490      2000
   macro avg     0.6138    0.6793    0.6357      2000
weighted avg     0.7580    0.7490    0.7522      2000



In [5]:

# Identify actual High cases predicted as Medium
high_missed_as_medium = X_test.loc[
    (y_test == "High") & (pd.Series(y_pred, index=y_test.index) == "Medium")
].copy()

high_missed_as_medium["Actual"] = y_test.loc[high_missed_as_medium.index]
high_missed_as_medium["Predicted"] = "Medium"

print("Actual High cases predicted as Medium:",
      len(high_missed_as_medium))

display(high_missed_as_medium.head(15))


Actual High cases predicted as Medium: 29


,Soil_Moisture,Temperature_C,Humidity,Crop_Type,Crop_Growth_Stage,Soil_Type,Actual,Predicted
4122,22.07,12.48,75.31,Rice,Flowering,Silt,High,Medium
4136,23.72,15.59,40.13,Cotton,Vegetative,Silt,High,Medium
4793,11.44,13.56,55.81,Cotton,Flowering,Clay,High,Medium
1485,44.91,39.58,60.18,Cotton,Vegetative,Clay,High,Medium
4577,12.25,15.15,27.66,Maize,Flowering,Clay,High,Medium
6637,17.02,31.54,47.67,Maize,Sowing,Loamy,High,Medium
4328,12.31,13.57,34.63,Sugarcane,Flowering,Clay,High,Medium
7837,18.86,20.49,35.68,Wheat,Flowering,Silt,High,Medium
5654,49.06,38.97,58.16,Potato,Vegetative,Clay,High,Medium
2960,22.01,27.58,83.39,Cotton,Vegetative,Clay,High,Medium


In [6]:

# Actual High cases that the model predicted correctly
high_correct = X_test.loc[
    (y_test == "High") &
    (pd.Series(y_pred, index=y_test.index) == "High")
].copy()

# Compare numeric feature distributions
numeric_features = [
    "Soil_Moisture",
    "Temperature_C",
    "Humidity",
]

comparison = pd.DataFrame({
    "Missed_High_Mean": high_missed_as_medium[numeric_features].mean(),
    "Correct_High_Mean": high_correct[numeric_features].mean(),
    "Missed_High_Median": high_missed_as_medium[numeric_features].median(),
    "Correct_High_Median": high_correct[numeric_features].median(),
})

print("Missed High cases:", len(high_missed_as_medium))
print("Correctly predicted High cases:", len(high_correct))

display(comparison.round(2))


Missed High cases: 29
Correctly predicted High cases: 38


,Missed_High_Mean,Correct_High_Mean,Missed_High_Median,Correct_High_Median
Soil_Moisture,24.29,16.95,21.45,16.15
Temperature_C,29.48,33.32,31.54,34.29
Humidity,58.02,57.90,55.81,61.06


In [7]:

categorical_features = [
    "Crop_Type",
    "Crop_Growth_Stage",
    "Soil_Type",
]

for feature in categorical_features:
    print(f"\n{'=' * 15} {feature} {'=' * 15}")

    missed_pct = (
        high_missed_as_medium[feature]
        .value_counts(normalize=True)
        .mul(100)
        .rename("Missed_High_%")
    )

    correct_pct = (
        high_correct[feature]
        .value_counts(normalize=True)
        .mul(100)
        .rename("Correct_High_%")
    )

    comparison = pd.concat(
        [missed_pct, correct_pct],
        axis=1,
    ).fillna(0)

    display(comparison.round(2))



=============== Crop_Type ===============


,Missed_High_%,Correct_High_%
Crop_Type,,
Maize,27.59,18.42
Cotton,20.69,10.53
Wheat,20.69,21.05
Sugarcane,17.24,15.79
Rice,6.90,18.42
Potato,6.90,15.79



=============== Crop_Growth_Stage ===============


,Missed_High_%,Correct_High_%
Crop_Growth_Stage,,
Vegetative,41.38,52.63
Flowering,37.93,47.37
Sowing,10.34,0.00
Harvest,10.34,0.00



=============== Soil_Type ===============


,Missed_High_%,Correct_High_%
Soil_Type,,
Silt,34.48,28.95
Clay,34.48,23.68
Sandy,20.69,31.58
Loamy,10.34,15.79


In [8]:

high_class = df.loc[df["Irrigation_Need"] == "High"].copy()

print("Total High-labelled rows:", len(high_class))

print("\nNumeric feature summary:")
display(
    high_class[
        ["Soil_Moisture", "Temperature_C", "Humidity"]
    ].describe().round(2)
)

print("\nCategorical feature distributions:")

for feature in ["Crop_Type", "Crop_Growth_Stage", "Soil_Type"]:
    print(f"\n{feature}:")
    display(high_class[feature].value_counts())


Total High-labelled rows: 336

Numeric feature summary:


,Soil_Moisture,Temperature_C,Humidity
count,336.00,336.00,336.00
mean,19.71,32.88,59.31
std,10.84,7.18,20.07
min,8.01,12.04,25.02
25%,12.24,30.50,41.61
50%,17.76,34.30,60.29
75%,22.10,38.24,75.62
max,64.22,41.99,94.70



Categorical feature distributions:

Crop_Type:


Crop_Type
Maize        68
Sugarcane    65
Cotton       55
Wheat        53
Potato       51
Rice         44
Name: count, dtype: int64


Crop_Growth_Stage:


Crop_Growth_Stage
Vegetative    164
Flowering     152
Harvest        11
Sowing          9
Name: count, dtype: int64


Soil_Type:


Soil_Type
Sandy    96
Clay     85
Silt     81
Loamy    74
Name: count, dtype: int64

In [9]:

# Build a table of actual High cases and their predictions
high_results = X_test.copy()
high_results["Actual"] = y_test
high_results["Predicted"] = y_pred

high_results = high_results[high_results["Actual"] == "High"].copy()

stage_analysis = high_results.groupby("Crop_Growth_Stage").agg(
    Actual_High_Count=("Actual", "size"),
    Correctly_Predicted_High=(
        "Predicted",
        lambda s: (s == "High").sum(),
    ),
)

stage_analysis["Recall_Percent"] = (
    stage_analysis["Correctly_Predicted_High"]
    / stage_analysis["Actual_High_Count"]
    * 100
)

display(stage_analysis.round(2))


,Actual_High_Count,Correctly_Predicted_High,Recall_Percent
Crop_Growth_Stage,,,
Flowering,29,18,62.07
Harvest,3,0,0.00
Sowing,3,0,0.00
Vegetative,32,20,62.50


In [10]:

high_results["Correct"] = (
    high_results["Actual"] == high_results["Predicted"]
)

stage_crop_errors = (
    high_results.groupby(["Crop_Growth_Stage", "Crop_Type"])
    .agg(
        Actual_High_Count=("Correct", "size"),
        Correct_Predictions=("Correct", "sum"),
    )
)

stage_crop_errors["Missed_High_Count"] = (
    stage_crop_errors["Actual_High_Count"]
    - stage_crop_errors["Correct_Predictions"]
)

display(
    stage_crop_errors
    .sort_values("Missed_High_Count", ascending=False)
    .head(15)
)


Actual_High_Count  Correct_Predictions  \
Crop_Growth_Stage Crop_Type                                           
Vegetative        Cotton                     7                    3   
                  Maize                      7                    4   
Flowering         Sugarcane                  7                    4   
Vegetative        Wheat                      7                    4   
Sowing            Maize                      2                    0   
Flowering         Rice                       3                    1   
                  Maize                      5                    3   
                  Wheat                      6                    4   
                  Cotton                     2                    1   
Harvest           Wheat                      1                    0   
                  Maize                      1                    0   
                  Cotton                     1                    0   
Flowering         Potato                     6                    5   
Sowing            Sugarcane                  1                    0   
Vegetative        Potato                     2                    1   

                             Missed_High_Count  
Crop_Growth_Stage Crop_Type                     
Vegetative        Cotton                     4  
                  Maize                      3  
Flowering         Sugarcane                  3  
Vegetative        Wheat                      3  
Sowing            Maize                      2  
Flowering         Rice                       2  
                  Maize                      2  
                  Wheat                      2  
                  Cotton                     1  
Harvest           Wheat                      1  
                  Maize                      1  
                  Cotton                     1  
Flowering         Potato                     1  
Sowing            Sugarcane                  1  
Vegetative        Potato                     1

In [11]:

from sklearn.metrics import accuracy_score

correct_count = int((y_test.to_numpy() == y_pred).sum())
error_count = len(y_test) - correct_count

print("Total test samples:", len(y_test))
print("Correct predictions:", correct_count)
print("Incorrect predictions:", error_count)
print("Accuracy (%):", round(accuracy_score(y_test, y_pred) * 100, 2))
print("Error rate (%):", round(error_count / len(y_test) * 100, 2))

print("\nHigh-class errors:")
print("Actual High samples:", int((y_test == "High").sum()))
print("High predicted correctly:", int(((y_test == "High") & (pd.Series(y_pred, index=y_test.index) == "High")).sum()))
print("High predicted as Medium:", int(((y_test == "High") & (pd.Series(y_pred, index=y_test.index) == "Medium")).sum()))


Total test samples: 2000
Correct predictions: 1498
Incorrect predictions: 502
Accuracy (%): 74.9
Error rate (%): 25.1

High-class errors:
Actual High samples: 67
High predicted correctly: 38
High predicted as Medium: 29



## Error Analysis — Findings

### Evaluation Summary
- Test samples: 2,000
- Correct predictions: 1,498
- Incorrect predictions: 502
- Accuracy: 74.90%
- Error rate: 25.10%

### High-Class Performance
- Actual High samples: 67
- Correctly predicted High samples: 38
- High-class recall: 56.72%
- High-class precision: 32.76%
- High-class F1-score: 41.53%
- Actual High cases predicted as Medium: 29

### Main Observations
1. The target variable is imbalanced; High represents only 3.36% of the full dataset.
2. The model performs better on Low than on High.
3. Some High cases are missed, especially as Medium predictions.
4. Growth-stage and crop-stage combinations show differences in error counts, but several groups have small sample sizes.
5. Numeric feature comparisons show differences between missed and correctly predicted High cases. These observations do not establish causation.

### Limitations
- Evaluation measures agreement with dataset labels, not verified real-world irrigation requirements.
- The underlying validity and provenance of target labels have not been established.
- Small subgroup sizes limit conclusions about individual crops and growth stages.
- The model should not be used as an automatic irrigation controller without agronomic validation.

### Conclusion
The Random Forest model achieves 74.90% accuracy, but High-class detection remains a limitation. Further work should investigate label quality, class imbalance, and model performance before considering deployment decisions.


In [13]:

# Inspect whether target labels show obvious relationships
target_audit = df.groupby("Irrigation_Need").agg(
    Sample_Count=("Irrigation_Need", "size"),
    Mean_Soil_Moisture=("Soil_Moisture", "mean"),
    Mean_Temperature=("Temperature_C", "mean"),
    Mean_Humidity=("Humidity", "mean"),
    Mean_Rainfall=("Rainfall_mm", "mean"),
    Mean_Previous_Irrigation=("Previous_Irrigation_mm", "mean"),
)

display(target_audit.round(2))


,Sample_Count,Mean_Soil_Moisture,Mean_Temperature,Mean_Humidity,Mean_Rainfall,Mean_Previous_Irrigation
Irrigation_Need,,,,,,
High,336,19.71,32.88,59.31,677.83,61.40
Low,5864,41.72,25.54,59.97,1346.28,59.53
Medium,3800,31.16,28.71,60.32,1158.60,60.25


In [14]:

features_to_audit = [
    "Soil_Moisture",
    "Temperature_C",
    "Rainfall_mm",
]

for feature in features_to_audit:
    print(f"\n{'=' * 12} {feature} {'=' * 12}")

    summary = df.groupby("Irrigation_Need")[feature].agg(
        ["min", "median", "max", "mean"]
    )

    display(summary.round(2))



============ Soil_Moisture ============


,min,median,max,mean
Irrigation_Need,,,,
High,8.01,17.76,64.22,19.71
Low,8.02,42.66,65.00,41.72
Medium,8.00,26.22,64.99,31.16



============ Temperature_C ============


,min,median,max,mean
Irrigation_Need,,,,
High,12.04,34.30,41.99,32.88
Low,12.00,24.91,42.00,25.54
Medium,12.02,30.45,42.00,28.71



============ Rainfall_mm ============


,min,median,max,mean
Irrigation_Need,,,,
High,2.28,254.39,2487.87,677.83
Low,1.85,1347.58,2499.58,1346.28
Medium,0.38,1139.18,2499.69,1158.60


In [15]:

# Exploratory diagnostic only — not a proposed irrigation rule
moisture_threshold = 26.22
temperature_threshold = 30.45

simple_rule = pd.Series("Medium", index=df.index)

simple_rule[
    (df["Soil_Moisture"] <= moisture_threshold)
    & (df["Temperature_C"] >= temperature_threshold)
] = "High"

simple_rule[
    (df["Soil_Moisture"] >  moisture_threshold)
    & (df["Temperature_C"] < temperature_threshold)
] = "Low"

print("Exploratory threshold-rule accuracy:")
print(f"{(simple_rule == df['Irrigation_Need']).mean() * 100:.2f}%")

print("\nRule predictions vs dataset labels:")
display(
    pd.crosstab(
        df["Irrigation_Need"],
        simple_rule,
        rownames=["Actual label"],
        colnames=["Simple rule"],
    )
)


Exploratory threshold-rule accuracy:
58.51%

Rule predictions vs dataset labels:


Simple rule,High,Low,Medium
Actual label,,,
High,217,1,118
Low,178,3440,2246
Medium,804,802,2194



## Exploratory Label-Pattern Check

A simple rule based on soil moisture and temperature achieved 58.51% accuracy against the dataset labels. The rule used exploratory thresholds derived from class medians.

This result does not establish that the labels are incorrect. The rule uses only two features, while the trained Random Forest uses six features and can model nonlinear relationships.

The result also does not validate the labels scientifically. The source and generation method of `Irrigation_Need` remain unknown.

### Conclusion
The label-generation process needs verification from the original dataset documentation or its author before making claims about real-world irrigation requirements.
